# Prämenien Daten von 2011-2026

Dieses Notebook lädt die Prämien Daten von https://opendata.swiss/de/dataset/health-insurance-premiums herunter und erstellt ein aggregiertes csv

In [11]:
import json
from pathlib import Path
import requests

In [12]:
DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw"
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

CKAN_API_URL = "https://ckan.opendata.swiss/api/3/action/package_show?id=health-insurance-premiums"
YEARS = list(range(2011, 2027))

## API Testen und Daten auslesen

In [13]:
response = requests.get(CKAN_API_URL, timeout=30)
response.raise_for_status()

package = response.json()["result"]
print(json.dumps(package["resources"], indent=4))

[
    {
        "access_services": [],
        "byte_size": 77830,
        "cache_last_updated": null,
        "cache_url": null,
        "coverage": "",
        "created": "2023-09-21T14:16:52.663450",
        "datastore_active": false,
        "datastore_contains_all_records_of_source_file": false,
        "description": {
            "fr": "",
            "en": "",
            "de": "",
            "it": ""
        },
        "display_name": {
            "fr": "",
            "de": "",
            "en": "to whom it may concern.pdf",
            "it": ""
        },
        "documentation": [],
        "download_url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL3RvIHdob20gaXQgbWF5IGNvbmNlcm4ucGRm",
        "format": "PDF",
        "hash": "",
        "id": "1dc14e05-4d27-49ff-92ad-7d27bbf58e5a",
        "identifier": "",
        "issued": "2020-09-22T00:00:00",
        "language": [],
        "last_modified": null,
        "license": "https://opendata.swiss/terms-of-us

In [14]:
resources = {
    resource.get("name").get("en", ""): {
        "name": resource.get("name").get("en", ""),
        "format": resource.get("format", ""),
        "url": resource.get("url", ""),
    }
    for resource in package["resources"]
}
print(json.dumps(resources, indent=4))

{
    "to whom it may concern.pdf": {
        "name": "to whom it may concern.pdf",
        "format": "PDF",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL3RvIHdob20gaXQgbWF5IGNvbmNlcm4ucGRm"
    },
    "Versichertenbestand_CH.xlsx": {
        "name": "Versichertenbestand_CH.xlsx",
        "format": "XLS",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL1ZlcnNpY2hlcnRlbmJlc3RhbmRfQ0gueGxzeA%3D%3D"
    },
    "Versichertenbestand_CH.csv": {
        "name": "Versichertenbestand_CH.csv",
        "format": "CSV",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL1ZlcnNpY2hlcnRlbmJlc3RhbmRfQ0guY3N2"
    },
    "Tarife.xlsx": {
        "name": "Tarife.xlsx",
        "format": "XLS",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL1RhcmlmZS54bHN4"
    },
    "Tarife.csv": {
        "name": "Tarife.csv",
        "format": "CSV",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1

In [15]:
premium_resources = {}

for resource in resources.values():
    name = resource["name"]

    if name.endswith(".zip") and name.startswith("Archiv_Praemien_"):
        year_str = name.removesuffix(".zip").split("_")[-1]
        if year_str.isdigit():
            premium_resources[int(year_str)] = resource

    elif name == "Prämien_CH.csv":
        premium_resources[2026] = resource

assert(len(premium_resources) == 16)
print(json.dumps(premium_resources, indent=4))

{
    "2026": {
        "name": "Pr\u00e4mien_CH.csv",
        "format": "CSV",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL1Byw6RtaWVuX0NILmNzdg%3D%3D"
    },
    "2025": {
        "name": "Archiv_Praemien_2025.zip",
        "format": "ZIP",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL0FyY2hpdl9QcmFlbWllbl8yMDI1LnppcA%3D%3D"
    },
    "2024": {
        "name": "Archiv_Praemien_2024.zip",
        "format": "ZIP",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL0FyY2hpdl9QcmFlbWllbl8yMDI0LnppcA%3D%3D"
    },
    "2023": {
        "name": "Archiv_Praemien_2023.zip",
        "format": "ZIP",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL0FyY2hpdl9QcmFlbWllbl8yMDIzLnppcA%3D%3D"
    },
    "2022": {
        "name": "Archiv_Praemien_2022.zip",
        "format": "ZIP",
        "url": "https://opendata.bagnet.ch/?r=/download&path=L1ByYWVtaWVuL0FyY2hpdl9QcmFlbWllbl8yMDIyLnppcA%3D%3

## Daten herunterladen

Die Daten bestehen aus den ZIP files der vergangen Jahren und eine CSV des aktuellen Jahres

In [ ]:
MB = 1024 * 1024
def download_file(url: str, target: Path, chunk_size: int = 1*MB) -> None:
    if target.exists():
        return

    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with target.open("wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                f.write(chunk)

premium_files = {}

for year, resource in premium_resources.items():
    suffix = resource["name"].split(".")[-1]
    target = RAW_DIR / f"Praemien_{year}.{suffix}"

    download_file(resource["url"], target)
    premium_files[year] = target